<a href="https://colab.research.google.com/github/VakitiSriVarsha/Gen-AI/blob/main/Knowledge_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"]=userdata.get('OPENAI_API_KEY')

In [ ]:
!pip install -q openai faiss-cpu pypdf markdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 21.8 MB/s eta 0:00:00


In [ ]:
import os
import faiss
import numpy as np
from pypdf import PdfReader
import markdown
from google.colab import files
from openai import OpenAI

client = OpenAI()

In [ ]:
def load_pdf(file_path):
    reader = PdfReader(file_path)
    return "\n".join(
        page.extract_text() for page in reader.pages if page.extract_text()
    )

def load_txt(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

def load_md(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return markdown.markdown(f.read())

In [ ]:
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap

    return chunks

In [ ]:
def embed_texts(texts):
    response = client.embeddings.create(
        model="text-embedding-3-large",
        input=texts
    )
    return np.array([item.embedding for item in response.data])

In [ ]:
class FAISSVectorStore:
    def __init__(self, dim):
        self.index = faiss.IndexFlatL2(dim)
        self.texts = []

    def add(self, embeddings, texts):
        self.index.add(embeddings)
        self.texts.extend(texts)

    def search(self, query_embedding, k=5):
        _, indices = self.index.search(
            np.array([query_embedding]), k
        )
        return [self.texts[i] for i in indices[0]]

In [ ]:
uploaded_files = files.upload()

store = FAISSVectorStore(dim=3072)

for filename in uploaded_files:
    print(f"Processing: {filename}")

    if filename.endswith(".pdf"):
        text = load_pdf(filename)
    elif filename.endswith(".txt"):
        text = load_txt(filename)
    elif filename.endswith(".md"):
        text = load_md(filename)
    else:
        print(f"Skipped unsupported file: {filename}")
        continue

    chunks = chunk_text(text)
    embeddings = embed_texts(chunks)
    store.add(embeddings, chunks)

print("✅ Documents indexed successfully")

Saving Week1-5.txt to Week1-5.txt
Processing: Week1-5.txt
✅ Documents indexed successfully


In [ ]:
def retrieve_context(question, k=5):
    query_embedding = embed_texts([question])[0]
    return store.search(query_embedding, k)

In [ ]:
def build_prompt(context_chunks, question):
    context = "\n\n".join(context_chunks)

    return f"""
You are a helpful knowledge assistant.
Use ONLY the information present in the context below.
If the answer is not present, say "I don't know."

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
def ask_question(question):
    context_chunks = retrieve_context(question)
    prompt = build_prompt(context_chunks, question)

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    return response.choices[0].message.content

In [ ]:
question = "What is the main topic covered in the document?"
answer = ask_question(question)

print("🤖 Answer:\n")
print(answer)

🤖 Answer:

The main topic covered in the document is how Riya discovered an old journal in a village library, learned about the importance of education and knowledge from Arun's writings, and was inspired to reopen and revitalize the library, which then became a central and lively part of the community.
